# Currency Converter with LangChain and Gradio

## 1. Setup API Key

In [47]:
# Used to securely store your API key
from google.colab import userdata
import os

# Retrieve the API keys from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["EXCHANGE_RATE_API_KEY"] = userdata.get("EXCHANGE_RATE_API_KEY")

## 2. Install Dependencies

Install the necessary libraries, including `langchain-groq` for integrating with Groq models.

In [48]:
!pip install -q langchain-groq

## 3. Import Libraries

Import all required modules from `langchain_groq`, `langchain_core.tools`, `requests`, and `ipywidgets`.

In [49]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

## 4. Initialize the Language Model

In [50]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

## 5. Define Tools for Currency Conversion

Two tools are defined:
- `get_conversion_factor`: Fetches the current exchange rate between two currencies using an external API.
- `convert`: Calculates the target currency value given a base amount and a conversion rate.

In [51]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated
import os

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  api_key = os.environ.get("EXCHANGE_RATE_API_KEY")
  if not api_key:
      raise ValueError("EXCHANGE_RATE_API_KEY not found in environment variables. Please set it in Colab Secrets.")
  url = f'https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate

In [52]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [53]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782864001,
 'time_last_update_utc': 'Wed, 01 Jul 2026 00:00:01 +0000',
 'time_next_update_unix': 1782950401,
 'time_next_update_utc': 'Thu, 02 Jul 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 94.6933}

In [54]:
convert.invoke({'base_currency_value':10, 'conversion_rate':94.6734})

946.734

### 6. Integrate Tools with LLM

In [55]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [56]:
messages = [HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr')]

In [57]:
ai_message = llm_with_tools.invoke(messages)

### 7. Process LLM Tool Calls

In [58]:
messages.append(ai_message)

In [59]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': '9zakfe667',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'ev2gwc5bv',
  'type': 'tool_call'}]

In [60]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [61]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10 usd to inr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '9zakfe667', 'function': {'arguments': '{"base_currency":"USD","target_currency":"INR"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'ev2gwc5bv', 'function': {'arguments': '{"base_currency_value":10}', 'name': 'convert'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 350, 'total_tokens': 388, 'completion_time': 0.087194738, 'completion_tokens_details': None, 'prompt_time': 0.033726772, 'prompt_tokens_details': None, 'queue_time': 0.175945001, 'total_time': 0.12092151}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1c43-3f8d-7400-9c18

### 8. Final LLM Response

After executing the tools and appending their outputs, the LLM can generate a final, coherent response based on the tool results.

In [62]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 94.6933. Based on this conversion factor, 10 USD is equivalent to 946.933 INR.'

## 9. Gradio UI for Currency Conversion

Now, let's create a more streamlined and easily shareable interface using `Gradio`. Gradio is excellent for building web-based demos for machine learning models and functions, and it's a natural fit for this kind of input-output pattern.

### 9.1 Install Gradio

First, we need to install the `gradio` library.

In [64]:
!pip install -q gradio

### 9.2 Import Gradio

Import the `gradio` library.

In [65]:
import gradio as gr

### 9.3 Define Gradio Interface Function

In [66]:
def gradio_currency_converter(base_currency: str, target_currency: str, base_value: float) -> str:
    try:
        b_currency = base_currency.upper()
        t_currency = target_currency.upper()

        # Get conversion factor using the tool
        conversion_response = get_conversion_factor.invoke({'base_currency': b_currency, 'target_currency': t_currency})

        if 'conversion_rate' in conversion_response:
            conversion_rate = conversion_response['conversion_rate']

            # Convert the amount using the tool
            converted_value = convert.invoke({'base_currency_value': base_value, 'conversion_rate': conversion_rate})

            result_text = f"Conversion Factor ({b_currency} to {t_currency}): {conversion_rate:.4f}\n"
            result_text += f"{base_value} {b_currency} is equal to {converted_value:.4f} {t_currency}"
            return result_text
        else:
            error_type = conversion_response.get('error-type', 'Unknown Error')
            return f"Could not retrieve conversion factor for {b_currency} to {t_currency}. Error: {error_type}"
    except Exception as e:
        return f"An error occurred: {e}"

### 9.4 Create and Launch the Gradio Interface

In [69]:
iface = gr.Interface(
    fn=gradio_currency_converter,
    inputs=[
        gr.Textbox(label="Base Currency (e.g., USD)", value="USD"),
        gr.Textbox(label="Target Currency (e.g., INR)", value="INR"),
        gr.Number(label="Amount to Convert", value=10.0)
    ],
    outputs=gr.Textbox(label="Conversion Result"),
    title="Currency Converter with LangChain and Gradio",
    description="Enter the base currency, target currency, and amount to get the conversion."
)

iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://44c47ed617bbffa7de.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/usr/local/lib/python3.12/di

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://44c47ed617bbffa7de.gradio.live
